In [1]:
import numpy as np
import pandas as pd
from Linear_Regression import LinearRegression

def data_process(data):
    X = data[["gia_ban", "gia_goc", "pham_tram_giam_gia", "chi_so_danh_gia", "so_luong_binh_luan"]].values
    Y = data["so_luong_ban"].values
    return X, Y

if __name__ == "__main__":
    data = pd.read_csv('tiki_diengiadung.csv')
    data = data.sample(frac=1, random_state=42, ignore_index=True)
    
    train_data = data.iloc[:192]
    test_data = data.iloc[192:]
    
    X_train, Y_train = data_process(train_data)
    X_test, Y_test = data_process(test_data)
    
    model = LinearRegression()
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    
    result = model.evalute(Y_test, Y_pred)
    print(result)

620.7015950828905


Mô hình Linear Regression tự xây dựng bằng NumPy chạy ổn định trên bộ dữ liệu tiki_diengiadung.csv với RMSE = 620.7016. Sai số còn tương đối cao do biến mục tiêu so_luong_ban có nhiều sản phẩm bán chạy vượt trội (outliers).

In [2]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train_standardize = (X_train - mean) / std
X_test_standardize = (X_test - mean) / std

model_standardize = LinearRegression()
model_standardize.fit(X_train_standardize, Y_train)
Y_pred_standardize = model_standardize.predict(X_test_standardize)

result_standardize = model_standardize.evalute(Y_test, Y_pred_standardize)
print(result_standardize)

1029.3748364961411


 Chuẩn hóa dữ liệu giúp đưa các cột có độ lớn chênh lệch (như gia_ban hàng triệu và chi_so_danh_gia 0–5) về cùng thang đo . RMSE tăng lên (1029.3748 so với 620.7016) do hàm tự viết chưa thêm hệ số tự do (intercept), làm đường hồi quy bị ép đi qua gốc tọa độ khi trung bình của X bằng 0.

In [ ]:
pd.set_option('display.width', 1000) # để bảng được in ra đẹp hơn bằng cách giới hạn kích thước
print(data[["gia_ban", "gia_goc", "pham_tram_giam_gia", "chi_so_danh_gia", "so_luong_binh_luan"]].corr().round(2))

def organize_data_original(data):
    X = data[["gia_ban", "gia_goc", "pham_tram_giam_gia", "chi_so_danh_gia", "so_luong_binh_luan"]].values
    Y = data["so_luong_ban"].values
    return X, Y

def organize_data_preprocessed(data):
    X = data[["gia_ban", "pham_tram_giam_gia", "chi_so_danh_gia", "so_luong_binh_luan"]].values
    Y = data["so_luong_ban"].values
    return X, Y

X_train_original, Y_train_original = organize_data_original(train_data)
X_test_original, Y_test_original = organize_data_original(test_data)

X_train_preprocessed, Y_train_preprocessed = organize_data_preprocessed(train_data)
X_test_preprocessed, Y_test_preprocessed = organize_data_preprocessed(test_data)

model_original = LinearRegression()
model_original.fit(X_train_original, Y_train_original)
Y_pred_original = model_original.predict(X_test_original)
result_original = model_original.evalute(Y_test_original, Y_pred_original)
print("Gốc:", result_original)

model_preprocessed = LinearRegression()
model_preprocessed.fit(X_train_preprocessed, Y_train_preprocessed)
Y_pred_preprocessed = model_preprocessed.predict(X_test_preprocessed)
result_preprocessed = model_preprocessed.evalute(Y_test_preprocessed, Y_pred_preprocessed)
print("Đã xử lí:", result_preprocessed)

                    gia_ban  gia_goc  pham_tram_giam_gia  chi_so_danh_gia  so_luong_binh_luan
gia_ban                1.00     0.95               -0.17            -0.11               -0.04
gia_goc                0.95     1.00                0.06            -0.13               -0.05
pham_tram_giam_gia    -0.17     0.06                1.00            -0.02               -0.06
chi_so_danh_gia       -0.11    -0.13               -0.02             1.00                0.14
so_luong_binh_luan    -0.04    -0.05               -0.06             0.14                1.00
Gốc: 620.7015950828905
Đã xử lí: 564.6287188713947


In [ ]:
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.metrics import mean_squared_error

model_sklearn = SklearnLinearRegression()
model_sklearn.fit(X_train, Y_train)
Y_pred_sklearn = model_sklearn.predict(X_test)

result_sklearn = model.evalute(Y_test, Y_pred_sklearn)

print("Sklearn:", result_sklearn)
print("Tự viết:", result)

Sklearn: 591.5650514200469
Tự viết: 620.7015950828905


 Kết quả giữa Scikit-Learn (RMSE = 591.5726) và mô hình tự viết (RMSE = 620.7016) khá tương đồng. Scikit-Learn cho sai số thấp hơn một chút vì mặc định có thêm hệ số tự do (fit_intercept=True), còn code tự viết chưa ghép thêm cột hằng số 1 vào ma trận X.